# Ge'ez Morphology: ByT5 Fine-Tuning on Google Colab / Kaggle

This notebook fine-tunes a subword-free **ByT5** model on Classical Ge'ez root-and-pattern morphology.
Run this with a **GPU runtime** (T4 GPU is free on Colab & Kaggle).

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets accelerate torch

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"BFloat16 Supported: {torch.cuda.is_bf16_supported()}")

### 2. Clone Repositories or Pull Dataset

In [ ]:
# Pull latest code and dataset
!git clone https://github.com/esubaalew/geez-morphology.git
!git clone https://github.com/esubaalew/geez-dataset.git

# Ensure output directories exist
!mkdir -p ./checkpoints

### 3. Load & Preprocess Morphological Training Pairs

In [ ]:
import json
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

MODEL_NAME = "google/byt5-small"
DATASET_PATH = "geez-dataset/output/synthetic_morphology_train.jsonl"
SAMPLE_LIMIT = 25000  # 25k pairs for fast convergence

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_LIMIT:
            break
        d = json.loads(line)
        feat = d.get("features", {})
        surface = d.get("surface_form", "")
        
        input_text = f"analyze: {surface}"
        target_text = f"root: {feat.get('root', '')} | tense: {feat.get('tense', '')} | person: {feat.get('person', '')} | gender: {feat.get('gender', '')} | number: {feat.get('number', '')} | stem: {feat.get('stem', '')}"
        samples.append({"input_text": input_text, "target_text": target_text})

print(f"Loaded {len(samples)} training pairs.")
print(f"Example input:  {samples[0]['input_text']}")
print(f"Example target: {samples[0]['target_text']}")

### 4. Dynamic Tokenization Dataset (Prevents Masking Bugs)

In [ ]:
class GeezDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        model_inputs = self.tokenizer(
            item["input_text"],
            max_length=self.max_len,
            truncation=True
        )
        labels = self.tokenizer(
            text_target=item["target_text"],
            max_length=self.max_len,
            truncation=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

split = int(len(samples) * 0.9)
train_ds = GeezDataset(samples[:split], tokenizer)
eval_ds = GeezDataset(samples[split:], tokenizer)
print(f"Train size: {len(train_ds)}, Eval size: {len(eval_ds)}")

### 5. Training (FP32 to Prevent T5/ByT5 Underflow)

In [ ]:
# NOTE: ByT5 / T5 models suffer from attention underflow in FP16, producing loss: 0.0 or nan.
# On T4 GPUs, FP32 (fp16=False) must be used. On A100/H100, bf16 can be used.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-3,
    num_train_epochs=3,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,               # CRITICAL: Keep False to prevent NaN loss
    bf16=use_bf16,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Starting fine-tuning with active gradient tracking...")
trainer.train()

### 6. Interactive Model Test

In [ ]:
model.eval()

def analyze_word(word: str):
    input_text = f"analyze: {word}"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=2,
            repetition_penalty=1.5,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test forms
test_words = ["ይቀትሉ", "ተቀተለት", "ይባርኩ", "ይቀድሱ", "ቀተለ"]
for tw in test_words:
    print(f"{tw} -> {analyze_word(tw)}")